# Bank AI Dispute Service — Project Blueprint

This notebook is a starter blueprint for a two-step bank credit-card dispute AI service.

## Design goals

- **Two-step LLM workflow**
  1. Classify the client message into one of 13 dispute categories.
  2. Retrieve only the relevant bank-policy pages/chunks and produce policy-grounded reasoning.
- **Model independent**
  - Business services do not depend on Azure OpenAI, OpenAI, Anthropic, or any other provider.
  - Provider-specific code is isolated behind a small interface.
- **Structured outputs**
  - Classification and reasoning are returned as validated Pydantic models.
- **Prompt separation**
  - Prompts live outside Python business logic.
  - Category descriptions live in configuration/domain data, not inside the output schema.
- **No hard-coded prompt paths in services**
  - Prompt loading is injected.
- **Auditable**
  - Keep category, model/provider, prompt version, policy references, and evidence separately traceable.

> This notebook uses dummy data and a `DummyStructuredGenerator`, so you can run the architecture locally without making an external model call.

## 1. Recommended project structure

```text
bank-dispute-service/
├── pyproject.toml
├── README.md
├── .env.example
├── app/
│   ├── main.py
│   ├── api/
│   │   ├── dependencies.py
│   │   └── routes/
│   │       └── disputes.py
│   ├── config/
│   │   ├── settings.py
│   │   └── dispute_categories.yaml
│   ├── domain/
│   │   └── dispute_category.py
│   ├── schemas/
│   │   ├── classification.py
│   │   ├── reasoning.py
│   │   └── dispute.py
│   ├── prompts/
│   │   ├── classification/
│   │   │   ├── system.md
│   │   │   └── task.md
│   │   └── reasoning/
│   │       ├── system.md
│   │       └── task.md
│   ├── prompt/
│   │   └── loader.py
│   ├── llm/
│   │   ├── base.py
│   │   └── providers/
│   │       ├── azure_openai.py
│   │       ├── openai.py
│   │       └── anthropic.py
│   ├── retrieval/
│   │   ├── base.py
│   │   └── policy_retriever.py
│   ├── services/
│   │   ├── dispute_classifier.py
│   │   └── dispute_reasoner.py
│   └── workflows/
│       └── dispute_workflow.py
└── tests/
    ├── test_classifier.py
    ├── test_reasoner.py
    └── test_workflow.py
```

The key dependency direction is:

```text
FastAPI
   ↓
DisputeWorkflow
   ↓
Classifier / Retriever / Reasoner
   ↓
interfaces (StructuredGenerator, PolicyRetriever)
   ↓
provider adapters / infrastructure
```

Business logic should never import a concrete Azure/OpenAI client.

## 2. Why two model calls

### Step 1 — Classification

Input:

```text
client message
+ 13 category names
+ 13 short category descriptions
```

Output:

```text
category
confidence
```

The full bank policy is **not** sent here.

### Retrieval

Use the selected category as a strong filter, then retrieve only the relevant policy sections/pages/chunks.

### Step 2 — Policy-grounded reasoning

Input:

```text
client message
+ selected category
+ selected category description
+ relevant bank policy excerpts only
```

Output:

```text
supported_by_policy
reasoning
evidence
missing_information
requires_reclassification
```

Important: Step 1 is a routing hypothesis. Step 2 validates it against policy instead of blindly explaining it.

## 3. Domain category enum

The enum is the stable machine-readable contract used by Python and downstream services.

The descriptions should **not** be embedded in the enum values.

In [ ]:
from enum import StrEnum

class DisputeCategory(StrEnum):
    GOODS_NOT_RECEIVED = "goods_not_received"
    SERVICES_NOT_RECEIVED = "services_not_received"
    CREDIT_NOT_PROCESSED = "credit_not_processed"
    DUPLICATE_PROCESSING = "duplicate_processing"
    INCORRECT_AMOUNT = "incorrect_amount"
    CANCELLED_RECURRING = "cancelled_recurring"
    CANCELLED_MERCHANDISE = "cancelled_merchandise"
    NOT_AS_DESCRIBED = "not_as_described"
    DAMAGED_OR_DEFECTIVE = "damaged_or_defective"
    PAID_BY_OTHER_MEANS = "paid_by_other_means"
    CASH_NOT_RECEIVED = "cash_not_received"
    CASH_AMOUNT_INCORRECT = "cash_amount_incorrect"
    OTHER = "other"

## 4. Category descriptions

Store business descriptions separately, for example in:

`app/config/dispute_categories.yaml`

Dummy content:

In [ ]:
DUMMY_CATEGORY_CONFIG = {
    "goods_not_received": {
        "code": "GNR",
        "name": "Goods Not Received",
        "description": (
            "The cardholder purchased merchandise but did not receive it "
            "by the expected delivery date."
        ),
        "version": "1.0",
    },
    "services_not_received": {
        "code": "SNR",
        "name": "Services Not Received",
        "description": "The cardholder paid for a service that was not provided.",
        "version": "1.0",
    },
    "credit_not_processed": {
        "code": "CNP",
        "name": "Credit Not Processed",
        "description": (
            "The merchant agreed to issue a refund or credit, but the credit "
            "has not been posted."
        ),
        "version": "1.0",
    },
    "duplicate_processing": {
        "code": "DUP",
        "name": "Duplicate Processing",
        "description": "The same transaction was processed more than once.",
        "version": "1.0",
    },
}

# In the real project, add all 13 categories to the YAML file.

A real YAML file would look like:

```yaml
goods_not_received:
  code: GNR
  name: Goods Not Received
  description: >
    The cardholder purchased merchandise but did not receive it
    by the expected delivery date.
  version: "1.0"

services_not_received:
  code: SNR
  name: Services Not Received
  description: >
    The cardholder paid for a service that was not provided.
  version: "1.0"
```

Why separate this from the prompt?

- Business wording can change without changing the Python contract.
- Category definitions can be reviewed/versioned independently.
- You can inject exactly the same category metadata into tests and prompts.

## 5. Structured output schemas

Use structured output instead of legacy function calling when the model is simply returning a classification/result and is not actually invoking application behavior.

In [ ]:
from pydantic import BaseModel, Field

class ClassificationResult(BaseModel):
    category: DisputeCategory
    confidence: float = Field(ge=0.0, le=1.0)

class PolicyEvidence(BaseModel):
    document_id: str
    page: int | None = None
    section: str | None = None
    evidence: str

class ReasoningResult(BaseModel):
    category: DisputeCategory
    supported_by_policy: bool
    reasoning: str
    evidence: list[PolicyEvidence] = Field(default_factory=list)
    missing_information: list[str] = Field(default_factory=list)
    requires_reclassification: bool = False

class DisputeResult(BaseModel):
    classification: ClassificationResult
    reasoning: ReasoningResult

## 6. Prompt files

### `prompts/classification/system.md`

```text
You are a credit-card dispute classification assistant.

Your task is to classify the client's dispute into exactly one supported
dispute category.

Use only:
1. The client information provided.
2. The supplied dispute-category definitions.

Do not invent facts.
Select the category that best matches the client's situation.
```

### `prompts/classification/task.md`

```text
CLIENT INFORMATION

{client_message}

SUPPORTED DISPUTE CATEGORIES

{category_descriptions}

Classify the dispute into exactly one supported dispute category.
```

### `prompts/reasoning/system.md`

```text
You are a credit-card dispute reasoning assistant.

A dispute category has already been proposed by a classification step.

Evaluate the client dispute against the supplied bank policy.

Use only:
1. Client information.
2. The proposed dispute category.
3. Supplied bank-policy excerpts.

The supplied policy is authoritative.
Do not invent bank rules, dates, requirements, or client facts.

If required information is missing, identify it rather than assuming it.

If the supplied policy does not support the proposed category, clearly indicate
that the category should be reconsidered.

Ground the explanation in the supplied policy evidence.
```

### `prompts/reasoning/task.md`

```text
CLIENT INFORMATION

{client_message}

PROPOSED DISPUTE CATEGORY

{category}

CATEGORY DESCRIPTION

{category_description}

RELEVANT BANK POLICY

{policy_context}

Evaluate whether the proposed category is supported by the supplied client
information and bank policy.
```

## 7. Prompt loader — services do not know filesystem paths

A service receives a prompt source/loader. It does not know that files live under
`app/prompts/...`.

In [ ]:
from typing import Protocol

class PromptSource(Protocol):
    def load(self, name: str) -> str:
        ...

class DictPromptSource:
    # Useful for notebook demos and unit tests.
    def __init__(self, prompts: dict[str, str]):
        self._prompts = prompts

    def load(self, name: str) -> str:
        return self._prompts[name]

In [ ]:
classification_prompts = DictPromptSource({
    "system": """
You are a credit-card dispute classification assistant.
Classify the client's dispute into exactly one supported dispute category.
Use only the client information and supplied category definitions.
Do not invent facts.
""".strip(),
    "task": """
CLIENT INFORMATION

{client_message}

SUPPORTED DISPUTE CATEGORIES

{category_descriptions}

Classify the dispute into exactly one supported dispute category.
""".strip(),
})

reasoning_prompts = DictPromptSource({
    "system": """
You are a credit-card dispute reasoning assistant.
Evaluate the proposed category against the supplied bank policy.
Use only the supplied client information, category, and bank policy.
Do not invent policy requirements or client facts.
If information is missing, identify it.
If policy does not support the proposed category, require reconsideration.
""".strip(),
    "task": """
CLIENT INFORMATION

{client_message}

PROPOSED DISPUTE CATEGORY

{category}

CATEGORY DESCRIPTION

{category_description}

RELEVANT BANK POLICY

{policy_context}

Evaluate whether the proposed category is supported by policy.
""".strip(),
})

In the real application, create a filesystem implementation once in the composition/configuration layer:

```python
from pathlib import Path

class FilePromptSource:
    def __init__(self, base_dir: Path):
        self.base_dir = base_dir

    def load(self, name: str) -> str:
        return (self.base_dir / name).read_text(encoding="utf-8")
```

Then inject a configured `FilePromptSource` into each service.

The service still does not hard-code `app/prompts/...`.

## 8. Model-independent AI interface

The classifier and reasoner depend only on the capability they need: generate a validated structured object.

They do **not** import `AzureOpenAIClient`.

In [ ]:
from typing import Protocol, TypeVar
from pydantic import BaseModel

T = TypeVar("T", bound=BaseModel)

class StructuredGenerator(Protocol):
    async def generate(
        self,
        *,
        system_prompt: str,
        user_prompt: str,
        response_model: type[T],
    ) -> T:
        ...

Provider adapters implement this interface:

```text
StructuredGenerator
       │
       ├── AzureOpenAIAdapter
       ├── OpenAIAdapter
       ├── AnthropicAdapter
       └── FutureProviderAdapter
```

Only the application composition/dependency-injection layer decides which adapter is used.

## 9. Dummy model implementation

This lets us run the entire architecture without Azure/OpenAI credentials.

In [ ]:
class DummyStructuredGenerator:
    async def generate(
        self,
        *,
        system_prompt: str,
        user_prompt: str,
        response_model: type[T],
    ) -> T:
        lower = user_prompt.lower()

        if response_model is ClassificationResult:
            if "not received" in lower or "still have not received" in lower:
                return ClassificationResult(
                    category=DisputeCategory.GOODS_NOT_RECEIVED,
                    confidence=0.95,
                )
            return ClassificationResult(
                category=DisputeCategory.OTHER,
                confidence=0.55,
            )

        if response_model is ReasoningResult:
            category = (
                DisputeCategory.GOODS_NOT_RECEIVED
                if "goods_not_received" in lower
                else DisputeCategory.OTHER
            )
            return ReasoningResult(
                category=category,
                supported_by_policy=True,
                reasoning=(
                    "The client states that the merchandise was not received "
                    "after the expected delivery date, and the supplied policy "
                    "excerpt identifies this situation as Goods Not Received."
                ),
                evidence=[
                    PolicyEvidence(
                        document_id="dummy-policy",
                        page=47,
                        section="Goods Not Received",
                        evidence="Expected delivery date has passed and merchandise was not received.",
                    )
                ],
                missing_information=[],
                requires_reclassification=False,
            )

        raise TypeError(f"Unsupported response model: {response_model}")

## 10. Category repository

The classifier needs all 13 descriptions. The reasoner needs the description for the selected category.

In [ ]:
class CategoryRepository:
    def __init__(self, categories: dict[str, dict]):
        self._categories = categories

    def as_prompt_text(self) -> str:
        blocks = []
        for category_id, data in self._categories.items():
            blocks.append(
                f"Category ID: {category_id}\n"
                f"Name: {data['name']}\n"
                f"Description: {data['description']}"
            )
        return "\n\n".join(blocks)

    def get_description(self, category: DisputeCategory) -> str:
        return self._categories[category.value]["description"]

## 11. Step 1 service — classifier

In [ ]:
class DisputeClassifier:
    def __init__(
        self,
        *,
        model: StructuredGenerator,
        prompts: PromptSource,
        categories: CategoryRepository,
    ):
        self._model = model
        self._prompts = prompts
        self._categories = categories

    async def classify(self, client_message: str) -> ClassificationResult:
        system_prompt = self._prompts.load("system")
        task_template = self._prompts.load("task")

        user_prompt = task_template.format(
            client_message=client_message,
            category_descriptions=self._categories.as_prompt_text(),
        )

        return await self._model.generate(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            response_model=ClassificationResult,
        )

## 12. Policy retrieval interface

Retrieval is also abstracted away from the workflow.

Later this could be:

- deterministic category → policy section mapping,
- Azure AI Search,
- vector database,
- hybrid search,
- SharePoint-backed indexing,
- another internal document service.

The workflow should not care.

In [ ]:
class PolicyRetriever(Protocol):
    async def retrieve(
        self,
        *,
        category: DisputeCategory,
        client_message: str,
    ) -> str:
        ...

In [ ]:
class DummyPolicyRetriever:
    async def retrieve(
        self,
        *,
        category: DisputeCategory,
        client_message: str,
    ) -> str:
        if category == DisputeCategory.GOODS_NOT_RECEIVED:
            return """
Document: Dispute Policy
Page: 47
Section: Goods Not Received

A Goods Not Received dispute may apply when merchandise was expected
by an agreed delivery date and the merchandise was not received.
""".strip()

        return """
Document: Dispute Policy
Page: 99
Section: Other

Review the case against the applicable dispute requirements.
""".strip()

### Recommended retrieval design

If the policy is well organized by dispute type, first filter deterministically:

```text
classification
    ↓
category-specific policy section(s)
    ↓
semantic/hybrid retrieval inside those sections
    ↓
top relevant chunks
```

This is usually better than searching a huge policy corpus with no category filter.

## 13. Step 2 service — policy-grounded reasoner

In [ ]:
class DisputeReasoner:
    def __init__(
        self,
        *,
        model: StructuredGenerator,
        prompts: PromptSource,
        categories: CategoryRepository,
    ):
        self._model = model
        self._prompts = prompts
        self._categories = categories

    async def reason(
        self,
        *,
        client_message: str,
        classification: ClassificationResult,
        policy_context: str,
    ) -> ReasoningResult:
        system_prompt = self._prompts.load("system")
        task_template = self._prompts.load("task")

        user_prompt = task_template.format(
            client_message=client_message,
            category=classification.category.value,
            category_description=self._categories.get_description(
                classification.category
            ),
            policy_context=policy_context,
        )

        return await self._model.generate(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            response_model=ReasoningResult,
        )

## 14. Workflow orchestration

The workflow owns the sequence:

1. classify,
2. retrieve,
3. reason.

It does not know which LLM vendor or retrieval technology is used.

In [ ]:
class DisputeWorkflow:
    def __init__(
        self,
        *,
        classifier: DisputeClassifier,
        retriever: PolicyRetriever,
        reasoner: DisputeReasoner,
    ):
        self._classifier = classifier
        self._retriever = retriever
        self._reasoner = reasoner

    async def process(self, client_message: str) -> DisputeResult:
        classification = await self._classifier.classify(client_message)

        policy_context = await self._retriever.retrieve(
            category=classification.category,
            client_message=client_message,
        )

        reasoning = await self._reasoner.reason(
            client_message=client_message,
            classification=classification,
            policy_context=policy_context,
        )

        return DisputeResult(
            classification=classification,
            reasoning=reasoning,
        )

## 15. Wire everything together

This is the only place where concrete implementations need to be selected.
In FastAPI, this logic would normally live in your dependency/configuration layer.

In [ ]:
categories = CategoryRepository(DUMMY_CATEGORY_CONFIG)
model = DummyStructuredGenerator()
retriever = DummyPolicyRetriever()

classifier = DisputeClassifier(
    model=model,
    prompts=classification_prompts,
    categories=categories,
)

reasoner = DisputeReasoner(
    model=model,
    prompts=reasoning_prompts,
    categories=categories,
)

workflow = DisputeWorkflow(
    classifier=classifier,
    retriever=retriever,
    reasoner=reasoner,
)

## 16. Run a dummy dispute

In [ ]:
client_message = """
I ordered a television from a merchant.
The merchant told me it would arrive on July 20.
It is now August 5 and I still have not received it.
""".strip()

result = await workflow.process(client_message)
result

Expected shape:

```json
{
  "classification": {
    "category": "goods_not_received",
    "confidence": 0.95
  },
  "reasoning": {
    "category": "goods_not_received",
    "supported_by_policy": true,
    "reasoning": "...",
    "evidence": [
      {
        "document_id": "dummy-policy",
        "page": 47,
        "section": "Goods Not Received",
        "evidence": "..."
      }
    ],
    "missing_information": [],
    "requires_reclassification": false
  }
}
```

## 17. FastAPI boundary

Keep FastAPI thin. The endpoint should validate the HTTP request, call the workflow, and return the result.

Example:

In [ ]:
# Illustrative code — normally this lives in app/api/routes/disputes.py

from pydantic import BaseModel

class DisputeRequest(BaseModel):
    client_message: str

# FastAPI example:
#
# @router.post("/disputes/analyze", response_model=DisputeResult)
# async def analyze_dispute(
#     request: DisputeRequest,
#     workflow: DisputeWorkflow = Depends(get_dispute_workflow),
# ) -> DisputeResult:
#     return await workflow.process(request.client_message)

## 18. Provider adapter template

Your business services should never change when you switch models.

A real provider adapter would look conceptually like:

```python
class SomeProviderAdapter:
    def __init__(self, provider_client, model_name: str):
        self._client = provider_client
        self._model_name = model_name

    async def generate(
        self,
        *,
        system_prompt: str,
        user_prompt: str,
        response_model: type[T],
    ) -> T:
        # Provider-specific SDK call here.
        # Convert provider response into response_model.
        ...
```

Then your application composition chooses:

```python
if settings.ai_provider == "azure_openai":
    model = AzureOpenAIAdapter(...)
elif settings.ai_provider == "openai":
    model = OpenAIAdapter(...)
elif settings.ai_provider == "anthropic":
    model = AnthropicAdapter(...)
```

Nothing inside `DisputeClassifier`, `DisputeReasoner`, or `DisputeWorkflow` changes.

## 19. Important implementation notes

### A. Structured output vs function calling

Use **structured output** for:

- Step 1 classification result,
- Step 2 reasoning/evidence result.

Use tool/function calling only when the model really needs to request an application action, for example:

```text
lookup_transaction(...)
retrieve_policy(...)
request_customer_document(...)
create_dispute_case(...)
```

You do not need function calling merely to force the category into a fixed schema.

### B. Do not trust Step 1 as final policy truth

Step 1 only sees:

```text
client message + category definitions
```

Therefore Step 2 should be able to say:

```text
requires_reclassification = true
```

or:

```text
missing_information = [...]
```

### C. Keep retrieval references

Do not flatten all retrieved policy text and lose source metadata in production.

Prefer retrieval objects such as:

```python
class PolicyChunk(BaseModel):
    document_id: str
    document_version: str
    page: int | None
    section: str | None
    text: str
```

Then format those chunks for the model while keeping their metadata for audit/logging.

### D. Do not log sensitive client data indiscriminately

For a bank service, define logging rules explicitly. Avoid dumping raw prompts and full client messages into standard application logs.

### E. Version things separately

Useful audit fields include:

```text
classification_prompt_version
reasoning_prompt_version
category_definition_version
policy_document_version
model_provider
model_name/deployment
retrieval_version
request/correlation ID
```

### F. Confidence is not a calibrated probability

An LLM-generated `0.94` should not automatically be interpreted as a true 94% probability.

If confidence will drive business thresholds, validate/calibrate it on labelled historical or synthetic evaluation data.

### G. Consider top candidates for ambiguous cases

For production evaluation, you may eventually prefer:

```python
primary_category
alternative_category | None
classification_signals
```

instead of relying solely on a raw confidence score.

Start simple first.

## 20. Suggested real-project implementation order

1. Create the package structure.
2. Define the 13 stable `DisputeCategory` values.
3. Create the 13 reviewed category descriptions.
4. Create Pydantic schemas.
5. Add prompt files.
6. Add `PromptSource`.
7. Add `StructuredGenerator`.
8. Implement classifier.
9. Implement deterministic/dummy policy retriever.
10. Implement reasoner.
11. Implement workflow.
12. Add unit tests using `DummyStructuredGenerator`.
13. Add the real model provider adapter.
14. Add the real policy retrieval implementation.
15. Add FastAPI dependency injection.
16. Add evaluation datasets and regression tests.
17. Add audit-safe observability.

## 21. Unit-testing strategy

Because the business services depend on interfaces, they are easy to test without calling a real model.

Examples:

```text
test_classifier.py
- returns a supported enum
- receives all category descriptions
- rejects/handles malformed model result

test_reasoner.py
- includes selected category
- includes retrieved policy
- captures missing information
- supports requires_reclassification

test_workflow.py
- classifier runs before retrieval
- retrieval uses classified category
- reasoner receives retrieved context
```

Avoid using live Azure/OpenAI calls in normal unit tests.

Use separate integration tests for provider SDKs.

## 22. Final architecture

```text
                    HTTP / FastAPI
                         │
                         ▼
                  DisputeWorkflow
                         │
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
     Classifier      Retriever       Reasoner
          │              │              │
          ▼              │              ▼
 StructuredGenerator     │      StructuredGenerator
          │              │              │
          └──── provider adapters ──────┘

Step 1:
client + 13 descriptions
        ↓
classification

Retrieval:
classification + client
        ↓
relevant policy only

Step 2:
client + classification + relevant policy
        ↓
grounded reasoning/evidence
```

This keeps prompts, domain definitions, business logic, retrieval, API code, and model-provider code separated.